In [1]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

2.6.0+cu124
12.4


In [2]:
# Uninstall
# !pip uninstall torch-scatter torch-sparse torch-cluster torch-spline-conv pyg-lib -y

In [3]:
# Update Torch
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [4]:
# Install PyG (automatic)
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch_geometric

In [5]:
# Verify instalation
import torch
import torch_geometric
import torch_scatter

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch_scatter.__version__)
print(torch_geometric.__version__)


2.6.0+cu124
12.4
True
2.1.2+pt26cu124
2.7.0


In [6]:
from model_PyG import *
from utils import *

In [7]:
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.transforms import Compose
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam

In [8]:
import torch_geometric
print(torch_geometric.__version__)

np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

2.7.0


### Utils

In [9]:
def info(data):
	print("Validate:\t {}".format(data.validate(raise_on_error=True)))
	print("Num. nodes:\t {}".format(data.num_nodes))
	print("Num. edges:\t {}".format(data.num_edges))
	print("Num. features:\t {}".format(data.num_node_features))
	print("Has isolated:\t {}".format(data.has_isolated_nodes()))
	print("Has loops:\t {}".format(data.has_self_loops()))
	print("Is directed:\t {}".format(data.is_directed()))
	print("Is undirected:\t {}".format(data.is_undirected()))
	print("{}".format(data.edge_index))
	print("{}".format(data.x))
	print("{}".format(data.edge_attr))

def compute_num_neg_samples(edge_index, num_nodes, ratio):
	E = edge_index.size(1)
	max_neg = num_nodes * num_nodes - E
	return min(int(ratio * E), max_neg)

def neg_ratio_schedule(epoch, max_epoch):
	start = 5.0
	end = 1.0
	return start - (start - end) * (epoch / max_epoch)

class EarlyStopping:
	def __init__(self, patience=5, delta=0, warmup=5, verbose=False):
		self.patience = patience
		self.delta = delta
		self.warmup = warmup
		self.verbose = verbose
		self.best_loss = None
		self.no_improvement_count = 0
		self.stop_training = False
	
	def check_early_stop(self, loss, epoch):
		if epoch >= self.warmup:
			if self.best_loss is None or loss < self.best_loss - self.delta:
				self.best_loss = loss
				self.no_improvement_count = 0
			else:
				self.no_improvement_count += 1
				if self.no_improvement_count >= self.patience:
					self.stop_training = True
					if self.verbose:
						print("Stopping early as no improvement has been observed.")

### Parameters

In [10]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Has transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

epochs = params["epochs"]
print("Epochs:\t", epochs)

lr = params["lr"]
print("Lr:\t", lr)

Exp:		 exp103
Raw data:	 flickr-lastfm
Methods:	 ['t-gae']
Has transformation: False
Dimension:	 32
Groups id:	 ['flickr-lastfm']
Subgroups id:	 {'flickr-lastfm': ['1', '2']}
Cuda:	 1
Epochs:	 500
Lr:	 0.0001


### Setup

In [11]:
# Parameters models

dataset = exp
encoders = ["GIN", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # Change
device = torch.device(f"cuda:{cuda}" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

NUM_HIDDEN_LAYERS = 12
HIDDEN_DIM = [1024] * NUM_HIDDEN_LAYERS + [1024]
output_feature_size = dimension # 128
# lr = 0.001
# epochs = 100

In [12]:
# Create datasets (groups and subgroups)

list_train_set = []

for group_id in groups_id:
	train_set = []
	for subgroup_id in subgroups_id[group_id]:
		train_set.append("{}_{}".format(group_id, subgroup_id))
	list_train_set.append(train_set)
list_train_set

[['flickr-lastfm_1', 'flickr-lastfm_2']]

### Create Data (PyG)

In [13]:
def add_edge_attributes(data: Data) -> Data:
	"""
	Compute edge attributes:
		1. Common Neighbors
		2. Jaccard Similarity
		3. Adamic-Adar
		4. Feature Similarity (cosine)

	The resulting edge_attr has shape [num_edges, 4].
	"""

	edge_index = data.edge_index
	x = data.x

	num_nodes = data.num_nodes
	num_edges = edge_index.size(1)

	# ---------------------------------------------------------
	# 1. Build an undirected NetworkX graph
	# ---------------------------------------------------------
	G = nx.Graph()
	G.add_nodes_from(range(num_nodes))

	edges = edge_index.t().tolist()

	# Remove self-loops and duplicate edges
	G.add_edges_from(
		(u, v) for u, v in edges if u != v
	)

	# ---------------------------------------------------------
	# 2. Compute node degrees
	# ---------------------------------------------------------
	degree = dict(G.degree())

	# ---------------------------------------------------------
	# 3. Compute feature similarity
	# ---------------------------------------------------------
	if x is not None:

		# Normalize node feature vectors
		x_norm = F.normalize(x.float(), p=2, dim=1)

		# Cosine similarity for each edge
		u = edge_index[0]
		v = edge_index[1]

		feature_sim = (x_norm[u] * x_norm[v]).sum(dim=1)

	else:
		feature_sim = torch.zeros(
			num_edges,
			dtype=torch.float
		)

	# ---------------------------------------------------------
	# 4. Compute structural edge attributes
	# ---------------------------------------------------------
	cn_values = []
	jaccard_values = []
	aa_values = []

	for u, v in edges:

		# Self-loops
		if u == v:
			cn = 0.0
			jaccard = 0.0
			aa = 0.0

		else:

			# Common neighbors
			common = set(nx.common_neighbors(G, u, v))
			cn = float(len(common))

			# Jaccard similarity
			neighbors_u = set(G.neighbors(u))
			neighbors_v = set(G.neighbors(v))

			union = neighbors_u | neighbors_v

			if len(union) > 0:
				jaccard = len(common) / len(union)
			else:
				jaccard = 0.0

			# Adamic-Adar
			aa = 0.0

			for z in common:
				deg_z = degree[z]

				if deg_z > 1:
					aa += 1.0 / torch.log(
						torch.tensor(float(deg_z))
					).item()

		cn_values.append(cn)
		jaccard_values.append(jaccard)
		aa_values.append(aa)

	# ---------------------------------------------------------
	# 5. Convert structural attributes to tensors
	# ---------------------------------------------------------
	cn = torch.tensor(
		cn_values,
		dtype=torch.float
	)

	jaccard = torch.tensor(
		jaccard_values,
		dtype=torch.float
	)

	adamic_adar = torch.tensor(
		aa_values,
		dtype=torch.float
	)

	# ---------------------------------------------------------
	# 6. Combine all edge attributes
	# ---------------------------------------------------------
	edge_attr = torch.stack(
		[
			cn,
			jaccard,
			adamic_adar,
			feature_sim
		],
		dim=1
	)

	# ---------------------------------------------------------
	# 7. Store in the PyG Data object
	# ---------------------------------------------------------
	data.edge_attr = edge_attr

	return data

def fit_edge_normalization(data1, data2, eps=1e-8):

	edge_attr = torch.cat(
		[data1.edge_attr, data2.edge_attr],
		dim=0
	).float()

	mean = edge_attr.mean(dim=0, keepdim=True)
	std = edge_attr.std(dim=0, keepdim=True)

	std = std.clamp_min(eps)

	return mean, std

def apply_edge_normalization(data, mean, std):

	data.edge_attr = (
		data.edge_attr.float() - mean
	) / std

	return data

In [14]:
def NormalizeNodeEdge(data1, data2):
	node_scaler = StandardScaler() # StandardScaler(with_mean=False)
	edge_scaler = StandardScaler() # StandardScaler(with_mean=False)

	# -------------------------
	# Node features
	# -------------------------
	x_all = np.vstack([
		data1.x.cpu().numpy(),
		data2.x.cpu().numpy()
	])

	node_scaler.fit(x_all)

	data1.x = torch.tensor(
		node_scaler.transform(data1.x.cpu().numpy()),
		dtype=torch.float
	)

	data2.x = torch.tensor(
		node_scaler.transform(data2.x.cpu().numpy()),
		dtype=torch.float
	)

	# -------------------------
	# Edge attributes
	# -------------------------
	edge_all = np.vstack([
		data1.edge_attr.cpu().numpy(),
		data2.edge_attr.cpu().numpy()
	])

	edge_scaler.fit(edge_all)

	data1.edge_attr = torch.tensor(
		edge_scaler.transform(data1.edge_attr.cpu().numpy()),
		dtype=torch.float
	)

	data2.edge_attr = torch.tensor(
		edge_scaler.transform(data2.edge_attr.cpu().numpy()),
		dtype=torch.float
	)

In [15]:
# Only for GIN
""" transform = Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(fill_value=1.0),
	T.ToDevice(device)
]) """

# For GIN and GINE
transform = T.Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(attr="edge_attr", fill_value="mean"),
	# T.AddLaplacianEigenvectorPE(k=3, attr_name=None, is_undirected=True),
	# T.AddRandomWalkPE(walk_length=8, attr_name=None),
	T.ToDevice(device)
])

In [16]:
print("Loading training datasets")
has_edge_attribures = False
list_train_loader = []

if raw_data_file == "ACM_DBLP":
	group_id = groups_id[0]
	
	# Read dataset
	b = np.load("data/ACM-DBLP.npz")

	# Node matching
	test_pairs = b["test_pairs"].astype(np.int32)
	pos_pairs = b["pos_pairs"].astype(np.int32)
	df_node_alignment_truth = pd.DataFrame(np.concatenate((test_pairs, pos_pairs), axis=0))
	df_node_alignment_truth

	# Create data
	subgroup_id = 1
	edge_index1 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x1 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

	subgroup_id = 2
	edge_index2 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x2 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

elif raw_data_file == "Douban Online_Offline":
	group_id = groups_id[0]
	
	# Read dataset
	a1, f1, a2, f2, test_pairs = load_douban()

	# Node matching
	test_pairs = torch.tensor(np.array(test_pairs, dtype=int)) - 1
	test_pairs = test_pairs.numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs.T)
	df_node_alignment_truth

	# Create data
	edge_index1, _ = dense_to_sparse(torch.from_numpy(a1.toarray()))
	x1 = torch.from_numpy(f1.toarray()).float()

	edge_index2, _= dense_to_sparse(torch.from_numpy(a2.toarray()))
	x2 = torch.from_numpy(f2.toarray()).float()

elif raw_data_file == "Cora1-Cora2":
	group_id = groups_id[0]
		
	# Read dataset
	loader = load_npz("data/cora.npz")
	data = loader["adj_matrix"]
	samples = data.shape[0]
	features = data.shape[1]
	values = data.data
	coo_data = data.tocoo()
	# indices = torch.LongTensor([coo_data.row, coo_data.col])
	indices = torch.from_numpy(np.array([coo_data.row, coo_data.col]))

	# Node matching
	N = 2708
	test_pairs = [[i, i] for i in range(N)]
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	node_features = loader["node_attr"]
	node_features = node_features.toarray()

	edge_index1 = torch.tensor(indices, dtype=torch.long)
	x1 = torch.tensor(node_features, dtype=torch.float)

	edge_index2 = torch.tensor(indices, dtype=torch.long)
	x2 = torch.tensor(node_features, dtype=torch.float)

elif raw_data_file == "flickr-lastfm":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-lastfm.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = torch.tensor(edges[0], dtype=torch.long)
	x1 = torch.tensor(node_features[0], dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0], dtype=torch.float)

	edge_index2 = torch.tensor(edges[1], dtype=torch.long)
	x2 = torch.tensor(node_features[1], dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1], dtype=torch.float)

	has_edge_attribures = True
	
# Create data
train_loader = {}

subgroup_id = 1
data1 = Data(x=x1, edge_index=edge_index1)
if has_edge_attribures:
    data1.edge_attr = edge_attr1
else:
	data1 = add_edge_attributes(data1)
print(subgroup_id)
info(data1)

subgroup_id = 2
data2 = Data(x=x2, edge_index=edge_index2)
if has_edge_attribures:
    data2.edge_attr = edge_attr2
else:
	data2 = add_edge_attributes(data2)
print(subgroup_id)
info(data2)

# Normalize
NormalizeNodeEdge(data1, data2)

subgroup_id = 1
data1 = transform(data1)
train_loader[f"{group_id}_{subgroup_id}"] = data1
print(subgroup_id)
info(data1)

subgroup_id = 2
data2 = transform(data2)
train_loader[f"{group_id}_{subgroup_id}"] = data2
print(subgroup_id)
info(data2)

list_train_loader.append(train_loader)

Loading training datasets
1
Validate:	 True
Num. nodes:	 12974
Num. edges:	 32298
Num. features:	 3
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[    0,     1,     1,  ..., 12972, 12972, 12973],
        [ 1041,  2032,  4507,  ..., 10023, 10882,  9137]])
tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.],
        ...,
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor([[0., 1., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        ...,
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.]])
2
Validate:	 True
Num. nodes:	 15436
Num. edges:	 32638
Num. features:	 3
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[    0,     1,     1,  ..., 15433, 15434, 15435],
        [ 4472,  1429,  1551,  ..., 14708,  2201,  2472]])
tensor([[0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        ...,
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 

/tmp/ipykernel_3936778/2403330950.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index1 = torch.tensor(edges[0], dtype=torch.long)
/tmp/ipykernel_3936778/2403330950.py:88: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x1 = torch.tensor(node_features[0], dtype=torch.float)
/tmp/ipykernel_3936778/2403330950.py:89: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_attr1 = torch.tensor(edges_attributes[0], dtype=torch.float)
/tmp/ipykernel_3936778/2403330950.py:91: UserWarning: To copy construct from a tensor, it is recommended to

Validate:	 True
Num. nodes:	 12974
Num. edges:	 45272
Num. features:	 3
Has isolated:	 False
Has loops:	 True
Is directed:	 False
Is undirected:	 True
tensor([[    0,     1,     1,  ..., 12971, 12972, 12973],
        [ 1041,  2032,  4507,  ..., 12971, 12972, 12973]], device='cuda:1')
tensor([[-0.2233, -0.0767,  0.2373],
        [-0.2233, -0.0767,  0.2373],
        [-0.2233, -0.0767,  0.2373],
        ...,
        [-0.2233, -0.0767,  0.2373],
        [-0.2233, 13.0439, -4.2134],
        [-0.2233, -0.0767,  0.2373]], device='cuda:1')
tensor([[-0.2759,  0.2835, -0.0607],
        [ 3.6242, -3.5269, -0.0607],
        [ 3.6242, -3.5269, -0.0607],
        ...,
        [-0.2759,  0.2835, -0.0607],
        [-0.2759,  0.2835, -0.0607],
        [-0.2759,  0.2835, -0.0607]], device='cuda:1')
2
Validate:	 True
Num. nodes:	 15436
Num. edges:	 48074
Num. features:	 3
Has isolated:	 False
Has loops:	 True
Is directed:	 False
Is undirected:	 True
tensor([[    0,     1,     1,  ..., 15433, 15434, 15435]

In [17]:
# Features details
# pd.DataFrame(data.x.cpu().numpy()).describe()

### Train

In [18]:
def fit_TGAE_subgraph_(encoder, dataset, no_samples, model, epochs, train_loader, lr, test_pairs=None):
	best_hitAtOne = 0
	best_hitAtFive = 0
	best_hitAtTen = 0
	best_hitAtFifty = 0
	list_loss = []

	optimizer = Adam(model.parameters(), lr=lr,weight_decay=5e-4)
	
	# Initialize early stopping
	patience = 10
	delta = 1e-4 # 1e-4
	warmup = 10
	early_stopping = EarlyStopping(patience=patience, delta=delta, warmup=warmup, verbose=True)

	loop_obj = tqdm(range(1, epochs + 1))
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		
		# Train
		model.train()
		loss = 0.0
		
		for ts in random.sample(train_set, k=len(train_set)): # shuffle train_set
			data = train_loader[ts]

			# Encoder
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
				# z = F.normalize(z, dim=1)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)

			# Positive edges
			pos_edge_index = data.edge_index
			
			# Negative edges
			# option 1
			neg_edge_index = negative_sampling(
				edge_index=data.edge_index,
				num_nodes=z.size(0),
				num_neg_samples=pos_edge_index.size(1), # Change 2 to other value if needed
				method="sparse"
			)

			# option 2 Negative edges (dynamic)
			""" ratio = neg_ratio_schedule(epoch, epochs)
			num_neg = compute_num_neg_samples(
				edge_index=edge_index,
				num_nodes=z.size(0),
				ratio=ratio
			)
			neg_edge_index = negative_sampling(
				edge_index=edge_index,
				num_nodes=z.size(0),
				num_neg_samples=num_neg,
				method="sparse"
			) """
			
			# Decoder
			# option 1
			pos_logits = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
			neg_logits = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
			
			# option 2
			""" pos_logits = F.cosine_similarity(
				z[pos_edge_index[0]],
				z[pos_edge_index[1]],
				dim=1
			)
			neg_logits = F.cosine_similarity(
				z[neg_edge_index[0]],
				z[neg_edge_index[1]],
				dim=1
			) """

			# Loss
			pos_labels = torch.ones_like(pos_logits)
			neg_labels = torch.zeros_like(neg_logits)

			# option 1
			""" loss_pos = binary_cross_entropy_with_logits(pos_logits, pos_labels)
			loss_neg = binary_cross_entropy_with_logits(neg_logits, neg_labels)
			loss += loss_pos + loss_neg """

			# option 2
			# num_pos = pos_edge_index.size(1)
			# num_neg = neg_edge_index.size(1)
			# pos_weight = torch.tensor([num_neg / num_pos], device=device)
			logits = torch.cat([pos_logits, neg_logits], dim=0)
			labels = torch.cat([pos_labels, neg_labels], dim=0)
			loss_temp = F.binary_cross_entropy_with_logits(logits, labels) #, pos_weight=pos_weight) # with pos_weight
			loss += loss_temp
			
		optimizer.zero_grad()
		loss = loss / no_samples
		loss.backward()
		optimizer.step()

		loop_obj.set_postfix_str(f"Loss: {loss.item():.4f}")
		list_loss.append(loss.item())

		# Check early stopping condition
		early_stopping.check_early_stop(loss.item(), epoch)
		if early_stopping.stop_training:
			print(f"Early stopping at epoch {epoch}")
			break

		# Evaluation (for firts dataset)
		""" model.eval()
		with torch.no_grad():
			keys = list(train_loader.keys())
			data1 = train_loader[keys[0]]
			data2 = train_loader[keys[1]]

			z1 = model(data1.x, data1.edge_index).detach()
			z2 = model(data2.x, data2.edge_index).detach()
			
			# Similarity matrix
			# option 1
			D = torch.cdist(z1, z2, 2)

			# option 2 (GPU problem)
			# D = 1 - F.cosine_similarity(z1.unsqueeze(1), z2.unsqueeze(0), dim=-1)

			# option 3 (Decoder cosine similarity)
			" "" z1n = F.normalize(z1, dim=1)
			z2n = F.normalize(z2, dim=1)
			D = 1 - (z1n @ z2n.T) " ""

			if dataset == "ACM_DBLP":
				test_idx = test_pairs[:, 0].astype(int)
				labels = test_pairs[:, 1].astype(int)
			else:
				test_idx = test_pairs[0, :].astype(int)
				labels = test_pairs[1, :].astype(int)
				
			hitAtOne = 0
			hitAtFive = 0
			hitAtTen = 0
			hitAtFifty = 0
			hitAtHundred = 0
			for i in range(len(test_idx)):
				dist_list = D[test_idx[i]]
				sorted_neighbors = torch.argsort(dist_list).cpu()
				label = labels[i]
				for j in range(100):
					if (sorted_neighbors[j].item() == label):
						if (j == 0):
							hitAtOne += 1
							hitAtFive += 1
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 4):
							hitAtFive += 1
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 9):
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 49):
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 100):
							hitAtHundred += 1
							break
			cur_hitAtOne = hitAtOne / len(test_idx)
			cur_hitAtFive = hitAtFive / len(test_idx)
			cur_hitAtTen = hitAtTen / len(test_idx)
			cur_hitAtFifty = hitAtFifty / len(test_idx)

			if(cur_hitAtOne > best_hitAtOne): best_hitAtOne = cur_hitAtOne
			if (cur_hitAtFive > best_hitAtFive): best_hitAtFive = cur_hitAtFive
			if (cur_hitAtTen > best_hitAtTen): best_hitAtTen = cur_hitAtTen
			if (cur_hitAtFifty > best_hitAtFifty): best_hitAtFifty = cur_hitAtFifty

	print("The best results achieved:")
	print("Hit@1: ", end="")
	print(best_hitAtOne)
	print("Hit@5: ", end="")
	print(best_hitAtFive)
	print("Hit@10: ", end="")
	print(best_hitAtTen)
	print("Hit@50: ", end="")
	print(best_hitAtFifty) """

	# Evaluation (for others dataset)
	dict_node_embeddings = {}
	model.eval()
	with torch.no_grad():
		for ts in train_set:
			data = train_loader[ts]
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			dict_node_embeddings[ts] = z.cpu().numpy()

	del loss, z
	# torch.cuda.synchronize()
	torch.cuda.empty_cache()
	
	return dict_node_embeddings, list_loss

In [19]:
def fit_TGAE_subgraph(model, encoder, exp, group_id, subgroups_id, train_loader, train_set, no_samples, epochs, lr):
	list_loss = []

	optimizer = Adam(model.parameters(), lr=lr, weight_decay=5e-4)
	
	# Initialize early stopping
	patience = 10
	delta = 1e-4
	warmup = 10
	early_stopping = EarlyStopping(patience=patience, delta=delta, warmup=warmup, verbose=True)

	loop_obj = tqdm(range(1, epochs + 1))
	
	# Train mode
	model.train() # Set the model to training mode (enables dropout and BN updates)
	
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		
		optimizer.zero_grad() # Reset accumulated gradients before backpropagation
		
		loss = torch.tensor(0.0, device=device)
		
		for ts in random.sample(train_set, k=len(train_set)): # shuffle train_set
			data = train_loader[ts]

			# Encoder
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
				# z = F.normalize(z, dim=1)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)

			# Positive edges
			pos_edge_index = data.edge_index
			
			# Negative edges
			# option 1
			neg_edge_index = negative_sampling(
				edge_index=data.edge_index,
				num_nodes=z.size(0),
				num_neg_samples=pos_edge_index.size(1), # Change 2 to other value if needed
				method="sparse"
			)

			# option 2 Negative edges (dynamic)
			""" ratio = neg_ratio_schedule(epoch, epochs)
			num_neg = compute_num_neg_samples(
				edge_index=edge_index,
				num_nodes=z.size(0),
				ratio=ratio
			)
			neg_edge_index = negative_sampling(
				edge_index=edge_index,
				num_nodes=z.size(0),
				num_neg_samples=num_neg,
				method="sparse"
			) """
			
			# Decoder
			# option 1
			pos_logits = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
			neg_logits = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
			
			# option 2
			""" pos_logits = F.cosine_similarity(
				z[pos_edge_index[0]],
				z[pos_edge_index[1]],
				dim=1
			)
			neg_logits = F.cosine_similarity(
				z[neg_edge_index[0]],
				z[neg_edge_index[1]],
				dim=1
			) """

			# Loss
			pos_labels = torch.ones_like(pos_logits)
			neg_labels = torch.zeros_like(neg_logits)

			# option 1
			""" loss_pos = binary_cross_entropy_with_logits(pos_logits, pos_labels)
			loss_neg = binary_cross_entropy_with_logits(neg_logits, neg_labels)
			loss += loss_pos + loss_neg """

			# option 2
			# num_pos = pos_edge_index.size(1)
			# num_neg = neg_edge_index.size(1)
			# pos_weight = torch.tensor([num_neg / num_pos], device=device)
			logits = torch.cat([pos_logits, neg_logits], dim=0)
			labels = torch.cat([pos_labels, neg_labels], dim=0)
			loss_temp = F.binary_cross_entropy_with_logits(logits, labels) #, pos_weight=pos_weight) # with pos_weight
			loss += loss_temp
			
		loss = loss / no_samples
		loss.backward() # Compute gradients of the loss w.r.t. model parameters
		optimizer.step() # Update model parameters using computed gradients

		loop_obj.set_postfix_str(f"Loss: {loss.item():.4f}")
		list_loss.append(loss.item())

		# Check early stopping condition
		early_stopping.check_early_stop(loss.item(), epoch)
		if early_stopping.stop_training:
			print(f"Early stopping at epoch {epoch}")
			break
		# torch.cuda.empty_cache()
	# Evaluation (for others dataset)
	list_df_node_embeddings = []

	model.eval() # Set the model to evaluation mode (disables dropout and BN updates)
	with torch.no_grad():
		for subgroup_id in subgroups_id:

			# Get node embeddings
			data = train_loader[f"{group_id}_{subgroup_id}"]
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			
			df_node_embeddings = pd.DataFrame(z.cpu().numpy())
			df_node_embeddings["subgroup_id"] = [subgroup_id] * len(df_node_embeddings)
			
			# Get node ids to replace original Id (this is only for MS data)
			""" df_nodes = pd.read_csv(f"experiments/output/{exp}/preprocessing/graphs_data/nodes_{group_id}_{subgroup_id}.csv")
			# idx, id, mz, rt, intensity_mean, intensity_cv
			df_node_embeddings.insert(0, "Id", df_nodes["id"].values) """

			df_node_embeddings.insert(0, "Id", range(len(df_node_embeddings))) # this is only for no MS data
			list_df_node_embeddings.append(df_node_embeddings)

		# Concat node embeddings
		df_node_embeddings_concat = pd.concat(list_df_node_embeddings, ignore_index=True)
		
		# Save node embeddings
		df_node_embeddings_concat.to_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", index=False)
	
	# Save loss
	np.save(f"experiments/output/{exp}/loss/{encoder}_{group_id}.npy", np.array(list_loss))

	del loss, z
	# torch.cuda.synchronize()
	torch.cuda.empty_cache()

In [20]:
list_train_set

[['flickr-lastfm_1', 'flickr-lastfm_2']]

In [21]:
list_train_loader

[{'flickr-lastfm_1': Data(x=[12974, 3], edge_index=[2, 45272], edge_attr=[45272, 3]),
  'flickr-lastfm_2': Data(x=[15436, 3], edge_index=[2, 48074], edge_attr=[48074, 3])}]

In [22]:
num_exp = 10 # Change

list_exp = []

for n_exp in range(num_exp):
	print(n_exp)
	# Train
	for encoder in encoders:
		for i, train_loader in enumerate(list_train_loader):
			np.random.seed(0)
			torch.manual_seed(0)
			torch.cuda.manual_seed(0)
			torch.cuda.manual_seed_all(0)
			torch.backends.cudnn.deterministic = True
			torch.backends.cudnn.benchmark = False

			group_id = groups_id[i]
			subgroups_id_ = subgroups_id[group_id]

			train_set = list_train_set[i]

			no_samples = len(train_set) # * (1 + 1)  # num datasets * num of samples by dataset 
			input_dim = train_loader[train_set[0]].num_node_features

			if encoder == "GIN":
				model = TGAE_GIN(NUM_HIDDEN_LAYERS,
							input_dim,
							HIDDEN_DIM,
							output_feature_size).to(device)
			elif encoder == "GINE":
				edge_dim = train_loader[train_set[0]].edge_attr.size(1)

				model = TGAE_GINE(NUM_HIDDEN_LAYERS,
							input_dim,
							HIDDEN_DIM,
							output_feature_size, edge_dim).to(device)

			print("Fitting model")
			print(encoder, dataset, train_set, lr, epochs, input_dim, output_feature_size, no_samples)

			# Calculate elapsed time
			start_time = time.perf_counter()
			
			fit_TGAE_subgraph(model, encoder, dataset, group_id, subgroups_id_, train_loader, train_set, no_samples, epochs, lr)

			end_time = time.perf_counter()
			execution_time = end_time - start_time

			list_exp.append([raw_data_file, encoder, execution_time])

	# Similarity analysis
	k = 1 # Change

	for encoder in encoders:
		for group_id in groups_id:
			df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
			# Id, 0, 1,	2, ..., subgroup_id

			subgroups_id_ = subgroups_id[group_id]

			# Calculate distance matrix (KNN)
			knn = NearestNeighbors(n_neighbors=k, metric="euclidean")

			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
			x = df_node_embeddings.iloc[:, 1:-1].values

			df_node_alignment = pd.DataFrame()
			df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

			for subgroup_id in subgroups_id_[1:]:
				df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
				y = df_node_embeddings.iloc[:, 1:-1].values

				knn.fit(y)
				distances, indices = knn.kneighbors(x)
				indices = indices.squeeze() # (N,)

				df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
				
				""" df_node_alignment[f"distances"] = distances
				avg_distances = df_node_alignment["distances"].mean()
				df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
				df_node_alignment """
			
			# Save previus node alignment
			df_node_alignment.to_csv(f"experiments/output/{exp}/filter_raw/node_alignment_{encoder}_{group_id}.csv")
			display(df_node_alignment)

			# Find node alignment for all datasets
			""" df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
			list_count_meta.append([encoder, group_id, len(df_node_alignment_filter), len(df_node_alignment)])
			# print(len(df_node_alignment_filter))
			# display(df_node_alignment_filter)
			print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")
			
			# Save common node id
			# common_node_id = df_node_alignment_filter.iloc[:, 0].values
			# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id)) """

			# Save common node id as .csv
			df_common_node_id = df_node_alignment.iloc[:, [0, 1]].copy() # df_common_node_id = df_node_alignment_filter.iloc[:, [0, 1]].copy()
			df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
			df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

	# Intersection and Union
	""" list_common_node_id = []
	for encoder in encoders:
		for group_id in groups_id:
			# Read common node
			common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
			list_common_node_id.append(common_node_id)

		common_node_id_union = list(set.union(*map(set, list_common_node_id)))
		common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))

		# Save common node id
		np.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))
		np.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) """

	# Network alignment (metrics)
	# list_accuracy = []
	# list_encoder = []

	for j, encoder in enumerate(encoders):
		for group_id in groups_id:
			# Read node alignment (after run analysis.ipynb)
			df_common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
			# df_common_node_id[[0, 1]] = np.sort(df_common_node_id[[0, 1]], axis=1)

			# Evaluation
			df_matching = df_common_node_id.merge(df_node_alignment_truth, on=[0, 1], how="inner")

			# Accuracy
			accuracy = len(df_matching) / len(df_node_alignment_truth)
			
			list_exp[2 * n_exp + j].append(accuracy)
			# list_accuracy.append(accuracy)
			# list_encoder.append(encoder)


0
Fitting model
GIN exp103 ['flickr-lastfm_1', 'flickr-lastfm_2'] 0.0001 500 3 32 2


Epoch: 141:  28%|██▊       | 140/500 [02:32<06:31,  1.09s/it, Loss: 0.4773]


Stopping early as no improvement has been observed.
Early stopping at epoch 141
Fitting model
GINE exp103 ['flickr-lastfm_1', 'flickr-lastfm_2'] 0.0001 500 3 32 2


Epoch: 303:  60%|██████    | 302/500 [05:53<03:52,  1.17s/it, Loss: 0.4265]


Stopping early as no improvement has been observed.
Early stopping at epoch 303


...

Stopping early as no improvement has been observed.
Early stopping at epoch 178
Fitting model
GINE exp103 ['flickr-lastfm_1', 'flickr-lastfm_2'] 0.0001 500 3 32 2


Epoch: 90:  18%|█▊        | 89/500 [01:45<08:05,  1.18s/it, Loss: 0.5102]


Stopping early as no improvement has been observed.
Early stopping at epoch 90


,flickr-lastfm_1,flickr-lastfm_2
0,0,6
1,1,12384
2,2,250
3,3,3967
4,4,1076
...,...,...
12969,12969,3476
12970,12970,13
12971,12971,278
12972,12972,562


,flickr-lastfm_1,flickr-lastfm_2
0,0,868
1,1,11531
2,2,1110
3,3,1633
4,4,310
...,...,...
12969,12969,310
12970,12970,61
12971,12971,9901
12972,12972,562


### Similarity analysis

### Network alignment (metrics)

In [23]:
df_node_alignment_truth

,0,1
0,8825,8449
1,11614,13650
2,11416,1628
3,4588,9105
4,12384,9143
...,...,...
447,7086,13322
448,3301,151
449,10899,3406
450,10003,3506


##### Summary

In [24]:
list_exp

[['flickr-lastfm', 'GIN', 153.2144630070543, 0.011061946902654867],
 ['flickr-lastfm', 'GINE', 354.73381646198686, 0.01327433628318584],
 ['flickr-lastfm', 'GIN', 153.38209268194623, 0.008849557522123894],
 ['flickr-lastfm', 'GINE', 117.39718721900135, 0.004424778761061947],
 ['flickr-lastfm', 'GIN', 190.54188635002356, 0.004424778761061947],
 ['flickr-lastfm', 'GINE', 146.85482813394628, 0.0022123893805309734],
 ['flickr-lastfm', 'GIN', 157.8696739509469, 0.008849557522123894],
 ['flickr-lastfm', 'GINE', 245.1461909620557, 0.011061946902654867],
 ['flickr-lastfm', 'GIN', 171.04450644005556, 0.008849557522123894],
 ['flickr-lastfm', 'GINE', 101.33207439596299, 0.004424778761061947],
 ['flickr-lastfm', 'GIN', 176.6203143770108, 0.004424778761061947],
 ['flickr-lastfm', 'GINE', 206.36338373005856, 0.008849557522123894],
 ['flickr-lastfm', 'GIN', 161.2181648259284, 0.008849557522123894],
 ['flickr-lastfm', 'GINE', 152.9853681399254, 0.0022123893805309734],
 ['flickr-lastfm', 'GIN', 179.96

In [25]:
print(NUM_HIDDEN_LAYERS)
print(HIDDEN_DIM)
print(output_feature_size)

12
[1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024]
32


In [26]:
df_summary = pd.DataFrame(list_exp)
df_summary.columns = ["Dataset", "Model", "Runtime", "Accuracy"]
df_summary = df_summary[["Dataset", "Model", "Accuracy", "Runtime"]]
df_summary.sort_values(by=["Model"], inplace=True)

In [27]:
df_text = df_summary.map(lambda x: f"{x}".replace(".", ","))
df_text

,Dataset,Model,Accuracy,Runtime
0,flickr-lastfm,GIN,"0,011061946902654867","153,2144630070543"
2,flickr-lastfm,GIN,"0,008849557522123894","153,38209268194623"
16,flickr-lastfm,GIN,"0,011061946902654867","114,9512213000562"
4,flickr-lastfm,GIN,"0,004424778761061947","190,54188635002356"
6,flickr-lastfm,GIN,"0,008849557522123894","157,8696739509469"
14,flickr-lastfm,GIN,"0,00663716814159292","179,96726419194601"
8,flickr-lastfm,GIN,"0,008849557522123894","171,04450644005556"
18,flickr-lastfm,GIN,"0,004424778761061947","196,51255551399663"
10,flickr-lastfm,GIN,"0,004424778761061947","176,6203143770108"
12,flickr-lastfm,GIN,"0,008849557522123894","161,2181648259284"
